# 데일리 매매 리뷰 — 오늘 진입한 전략들의 "생각"

이 노트북은 `quant/control/trade_review.py`(순수 조립)와 `quant/apps/report_cli.py`의
봉 조회 헬퍼를 그대로 불러와 쓴다 — 로직을 여기서 다시 만들지 않는다. 상시 페이지
(`report_cli trade-review`가 만드는 `out/YYYY/MM/DD/{market}_trade_review.html`)와
같은 데이터, 다른 렌더러다.

**차트는 Plotly가 아니라 matplotlib이다** — 이 저장소에 Plotly 파이썬 패키지가
설치돼 있지 않고(상시 HTML 페이지는 CDN에서 JS로 불러와 설치가 필요 없다), 이
노트북 하나를 위해 새 의존성을 추가하는 대신 이미 의존성에 있고 다른 노트북
(`portfolio_analysis.ipynb`)도 쓰는 matplotlib으로 같은 요소(캔들·BUY/SELL·
손절/목표 밴드·보유구간·MA60)를 그린다.

숫자를 지어내지 않는다 — 봉을 못 구하면 차트 없이 표만 나온다.

In [ ]:
# ── 파라미터 셀 ──────────────────────────────────────────────────────────
MARKET = "KR"            # "KR" | "US"
DATE = "2026-09-07"       # YYYY-MM-DD
ROOT = "."                # 저장소 루트(이 노트북을 backtest/에서 그대로 실행한다고 가정)
LEDGER_PATH = None        # None이면 {ROOT}/data/state/trades.jsonl

In [ ]:
import sys
from datetime import date
from pathlib import Path

ROOT_PATH = Path(ROOT).resolve()
if str(ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(ROOT_PATH))

from quant.apps.config import DEFAULT_SETTINGS_PATH, load_settings
from quant.apps.report_cli import _fetch_trade_review_bars, _paths
from quant.control.ledger import DEFAULT_LEDGER_PATH, load_trades, trades_in_session
from quant.control.trade_review import build_trade_review, format_telegram_line

on = date.fromisoformat(DATE)
ledger_path = Path(LEDGER_PATH) if LEDGER_PATH else ROOT_PATH / DEFAULT_LEDGER_PATH
trades = load_trades(ledger_path)
session_fills = trades_in_session(trades, MARKET, on)
print(f"{MARKET} {on.isoformat()} 체결 {len(session_fills)}건 (원장 {ledger_path})")

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_KOREAN_FONT_CANDIDATES = [
    "AppleGothic", "Apple SD Gothic Neo", "NanumGothic", "Malgun Gothic",
    "Noto Sans CJK KR", "Noto Sans KR",
]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_KO_FONT = next((f for f in _KOREAN_FONT_CANDIDATES if f in _available_fonts), None)
if _KO_FONT:
    plt.rcParams["font.family"] = _KO_FONT
    plt.rcParams["axes.unicode_minus"] = False
else:
    print("한글 폰트를 찾지 못함 — 차트 라벨이 네모(tofu)로 보일 수 있다(텍스트 출력은 정상).")


In [ ]:
_, out_root, cache_dir, _ = _paths(ROOT_PATH)
settings_path = ROOT_PATH / DEFAULT_SETTINGS_PATH
settings = load_settings(str(settings_path)) if settings_path.exists() else load_settings()

symbols = sorted({str(f.get("symbol")) for f in session_fills if f.get("symbol")})
bars_by_symbol, bar_meta = ({}, {})
if symbols:
    bars_by_symbol, bar_meta = _fetch_trade_review_bars(symbols, MARKET, cache_dir)
    print(f"봉 확보: {len(bars_by_symbol)}/{len(symbols)}종목 — 출처: "
          + ", ".join(sorted({m['source'] for m in bar_meta.values()})) if bar_meta else "봉 없음")

review = build_trade_review(
    trades, bars_by_symbol, settings.strategies, MARKET, on, risk_params=settings.risk,
    bar_meta_by_symbol=bar_meta,
)
print(format_telegram_line(review) or "오늘 진입 체결 없음 — 표시할 카드가 없다.")

## 1. 오늘 요약

In [ ]:
import pandas as pd

totals = review["summary"]["totals"]
display(pd.DataFrame([totals]))

per_strategy = review["summary"]["per_strategy"]
if per_strategy:
    df = pd.DataFrame.from_dict(per_strategy, orient="index")
    display(df.sort_index())
else:
    print("전략별 종결 트립 없음.")

## 2. 종목별 카드 — 캔들 · BUY/SELL · 손절/목표 밴드 · 생각

노란 배경 = 보유 구간(진입→청산), 빨강 = 진입가→손절, 초록 = 진입가→목표,
주황 선 = MA60(scalp_1m류 청산 규칙이 실제로 보는 값). 봉이 없으면 차트를
생략하고 표만 낸다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime


def plot_trade(g: dict) -> None:
    """카드 하나 — matplotlib 캔들 + BUY/SELL + 밴드 + MA60."""
    bars = g.get("bars_window") or []
    if len(bars) < 2:
        print(f"  (봉 없음 — {g.get('bars_source') or '출처 미상'})")
        return

    ts = [datetime.fromisoformat(b["ts"]) for b in bars]
    o = [b["open"] for b in bars]
    h = [b["high"] for b in bars]
    l = [b["low"] for b in bars]
    c = [b["close"] for b in bars]
    ma60 = [b.get("ma60") for b in bars]

    fig, ax = plt.subplots(figsize=(9, 3.6))
    width = (ts[1] - ts[0]) * 0.6 if len(ts) > 1 else None
    for t, oo, hh, ll, cc in zip(ts, o, h, l, c):
        color = "#C1272D" if cc >= oo else "#2F5FC4"  # KR 관례: 상승=적색, 하락=청색
        ax.plot([t, t], [ll, hh], color=color, linewidth=1)
        if width is not None:
            ax.add_patch(plt.Rectangle(
                (mdates.date2num(t) - width.total_seconds() / 86400 / 2, min(oo, cc)),
                width.total_seconds() / 86400, max(abs(cc - oo), 1e-9),
                color=color,
            ))
    if any(v is not None for v in ma60):
        ax.plot(ts, ma60, color="#D98A1E", linewidth=1.3, label="MA60")

    entry_ts = datetime.fromisoformat(g["entries"][0]["ts"]) if g["entries"] else None
    exit_ts = datetime.fromisoformat(g["exits"][-1]["ts"]) if g["exits"] else ts[-1]
    if entry_ts is not None:
        ax.axvspan(entry_ts, exit_ts, color="#E6B414", alpha=0.12, label="보유 구간")
        entry_price = g["entry_price"]
        band = g["band"]
        if band.get("target") is not None:
            ax.axhspan(min(entry_price, band["target"]), max(entry_price, band["target"]),
                       xmin=0, xmax=1, color="#0A7D33", alpha=0.10)
        if band.get("stop") is not None:
            ax.axhspan(min(entry_price, band["stop"]), max(entry_price, band["stop"]),
                       xmin=0, xmax=1, color="#C1121F", alpha=0.10)
        ax.scatter([entry_ts], [entry_price], marker="^", color="#0052FF", s=80, zorder=5, label="BUY")
    if g["exits"]:
        ax.scatter([exit_ts], [g["exits"][-1]["price"]], marker="v", color="#A6570A", s=80, zorder=5, label="SELL")

    ax.set_title(f"{g['symbol']} · {g['strategy_id']} · {g.get('bars_source', '?')}", fontsize=10)
    ax.legend(loc="upper left", fontsize=7, framealpha=0.6)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


for g in review["groups"]:
    status = "종결" if g["status"] == "closed" else "보유중"
    pnl_txt = (f"{g['pnl']:+.0f} ({g['pnl_bp']:+.1f}bp)" if g["pnl_known"]
               else ("손익미상" if g["status"] == "closed" else "-"))
    print(f"### {g['symbol']} · {g['strategy_id']} · {status} · {pnl_txt}")
    print(f"  진입 사유: {g['thinking']['entry_reason']}")
    if g["thinking"]["exit_reason"]:
        print(f"  청산 사유: {g['thinking']['exit_reason']}")
    print(f"  밴드: 손절={g['band']['stop']} 목표={g['band']['target']} (출처: {g['band']['band_source']})")
    plot_trade(g)
    print()